# Emissions-allocation pipeline demonstration

This notebook is a guide of the emissions-allocation pipeline applied to two ships. It follows the stages in `docs/METHODOLOGY.md` and shows the datasets, checkpoints, validation assertions, and outputs that pass between them.

The implementation remains in `src/emissions_allocation/` and its SQL stages. This notebook deliberately does not call Global Fishing Watch, alter input data, or reimplement a processing step. 

## Table of contents

1. [Configuration and imports](#1-configuration-and-imports)
2. [Pipeline map and artifact inventory](#2-pipeline-map-and-artifact-inventory)
3. [Select pilot vessels](#3-stage-0-select-vessels)
4. [Acquire ship activity and identify unique ships](#4-stage-1-acquire-ship-activity-and-identify-unique-ships)
5. [Complete registry data and ship specifications](#5-stage-2-complete-registry-data-and-ship-specifications)
6. [Assign fuel type and emission factors](#6-stage-3-assign-fuel-type-and-emission-factors)
7. [Calculate scenario CO2 emissions](#7-stage-4-calculate-CO2-emissions)
8. [Classify international ships and allocate emissions](#8-stage-5-classify-international-ships-and-allocate-emissions)
9. [Establish baselines and compute allocation impacts](#9-stages-6-7-establish-baselines-and-compute-allocation-impacts)
10. [Sensitivity and validation](#stage-8-sensitivity-and-validation)

The cells below inspect persisted artifacts in their processing order. They are intended for explanation and review, not as an alternative way to run the study.

## 1. Configuration and imports

Set the project root, import the production modules, and locate the configured interim and output directories. The configuration is read exactly as it is by the pipeline; no notebook-specific settings are introduced.

**Pipeline:** `src/emissions_allocation/config.py` resolves the study configuration and project paths.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import json

import numpy as np
import pandas as pd
from IPython.display import display

from emissions_allocation import activity, allocation, load_config

FIG_SINGLE = (6.5, 4.0)
FIG_MAP = (7.0, 6.0)
FIG_PANEL = (13.0, 4.0)
FIG_GRID = (13.0, 9.0)
DPI_SAVE = 300

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

cfg = load_config()
INTERIM = cfg.path("interim")
OUT = cfg.path("out")

def read_checkpoint(name: str) -> pd.DataFrame:
    """Read one persisted pipeline checkpoint.

    Parameters
    ----------
    name : str
        Filename relative to ``data/interim``.

    Returns
    -------
    pandas.DataFrame
        The checkpoint at its documented grain.

    Raises
    ------
    FileNotFoundError
        If the pipeline has not created the requested artifact.
    """
    checkpoint = INTERIM / name
    if not checkpoint.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint}. Run the pipeline before this audit notebook.")
    return pd.read_parquet(checkpoint)

print(f"Study period: {cfg.start_date} to {cfg.end_date} ({cfg.elapsed_hours:,} elapsed hours)")
print(f"Configured vessels: {[v.imo for v in cfg]}")
print(f"Configured scenarios: {len(cfg.scenarios())}")


Study period: 2017-01-01 to 2024-12-31 (70,128 elapsed hours)
Configured vessels: ['9516454', '9277802']
Configured scenarios: 8


## 2. Pipeline map and artifact inventory

The pipeline is staged:

- GFW 4Wings `public-global-presence:latest` responses from `POST /v3/4wings/report` provide hourly positions and vessel identity. GFW Events `public-global-port-visits-events:latest` responses from `GET /v3/events` provide port visits. The GFW Vessels API supplies identity lookups. These raw JSON responses become `vessel_hour`, `port_call`, `voyage_leg`, and `coverage` checkpoints.
- Equasis vessel and company records supply flag, tonnage, deadweight, ship type, owner, ISM manager, and commercial manager. IMO Fourth GHG Study tables and configuration complete the vessel specifications.
- Hourly positions are joined to Marine Regions World EEZ v12, World High Seas v2, MARPOL Annex VI Regulation 14 ECA polygons, and Marine and Land Zones v4. Port events and voyage legs supply the EU-to-EU-leg flag. These joins, together with IMO fuel and emission-factor tables, produce `fuel_assignment`.
- `vessel_hour`, vessel specifications, `fuel_assignment`, and IMO Fourth GHG Study operating-mode and engine-load tables produce scenario-keyed hourly and annual CO₂ emissions.
- Annual emissions are combined with Equasis allocation keys and the EEZ/high-seas domestic test to produce country allocations. Those allocations join the Global Carbon Budget 2025 *National Fossil Carbon Emissions* workbook to produce impacts.

**Pipeline:** `src/emissions_allocation/db.py` and `src/emissions_allocation/sql/00_register_views.sql` provide the DuckDB hand-off layer.


In [2]:
expected = [
    "vessel_hour_{imo}.parquet", "port_call_{imo}.parquet", "voyage_leg_{imo}.parquet",
    "coverage_{imo}.parquet", "fuel_assignment_{imo}.parquet", "emissions_hour_{imo}.parquet",
    "emissions_year_{imo}.parquet",
]
artifact_descriptions = {
    "vessel_hour_{imo}.parquet": "Hourly vessel positions, derived speed, and activity flags.",
    "port_call_{imo}.parquet": "GFW port visits with timestamps, port identity, country, and anchorage.",
    "voyage_leg_{imo}.parquet": "Derived port-to-port journeys with distance, duration, and EU-to-EU status.",
    "coverage_{imo}.parquet": "Observed and expected vessel-hours by year, before gap treatment.",
    "fuel_assignment_{imo}.parquet": "Fuel type and emission-factor inputs for each vessel-hour.",
    "emissions_hour_{imo}.parquet": "Operating mode, load, fuel consumption, and CO2 for each vessel-hour and scenario.",
    "emissions_year_{imo}.parquet": "Annual vessel CO2 totals for every power and smoothing scenario.",
    "baseline.parquet": "Global Carbon Budget national territorial CO2 baselines by year.",
    "allocation.parquet": "Annual vessel CO2 assigned by country, allocation option, and scenario.",
    "impacts.parquet": "Allocated emissions expressed as increments to national carbon budgets.",
    "scenario_spread.parquet": "Minimum and maximum impact across the configured scenario space.",
}
rows = []
for template in expected:
    for vessel in cfg:
        filename = template.format(imo=vessel.imo)
        artifact = INTERIM / filename
        rows.append({
            "artifact": filename,
            "stage": template.split("_")[0],
            "description": artifact_descriptions[template],
            "present": artifact.exists(),
            "bytes": artifact.stat().st_size if artifact.exists() else np.nan,
        })
for filename in ["baseline.parquet", "allocation.parquet", "impacts.parquet", "scenario_spread.parquet"]:
    artifact = INTERIM / filename
    rows.append({"artifact": filename, "stage": "final", "description": artifact_descriptions[filename],
                 "present": artifact.exists(), "bytes": artifact.stat().st_size if artifact.exists() else np.nan})
inventory = pd.DataFrame(rows)
assert inventory.present.all(), "One or more pipeline checkpoints are missing."
display(inventory.style.format({"bytes": "{:,.0f}"}).set_caption("Persisted pipeline artifacts required by this audit"))


,artifact,stage,description,present,bytes
0,vessel_hour_9516454.parquet,vessel,"Hourly vessel positions, derived speed, and activity flags.",True,"3,265,723"
1,vessel_hour_9277802.parquet,vessel,"Hourly vessel positions, derived speed, and activity flags.",True,"3,808,461"
2,port_call_9516454.parquet,port,"GFW port visits with timestamps, port identity, country, and anchorage.",True,"56,082"
3,port_call_9277802.parquet,port,"GFW port visits with timestamps, port identity, country, and anchorage.",True,"87,436"
4,voyage_leg_9516454.parquet,voyage,"Derived port-to-port journeys with distance, duration, and EU-to-EU status.",True,"20,575"
5,voyage_leg_9277802.parquet,voyage,"Derived port-to-port journeys with distance, duration, and EU-to-EU status.",True,"31,771"
6,coverage_9516454.parquet,coverage,"Observed and expected vessel-hours by year, before gap treatment.",True,"6,198"
7,coverage_9277802.parquet,coverage,"Observed and expected vessel-hours by year, before gap treatment.",True,"6,175"
8,fuel_assignment_9516454.parquet,fuel,Fuel type and emission-factor inputs for each vessel-hour.,True,"627,664"
9,fuel_assignment_9277802.parquet,fuel,Fuel type and emission-factor inputs for each vessel-hour.,True,"628,937"


## 3. Stage 0: Select vessels

**Purpose:** Define the pilot vessels and their stable IMO identifiers before requesting activity data.

**Dataset and hand-off:** The selection pool begins with GFW vessel identity and presence data. Equasis verifies the flag, deadweight, and owner-country criterion. Versioned configuration records the selected vessel's IMO, name history, flag, type, and specifications. Retained GFW discovery responses document why the second vessel was selected.

**Pipeline:** `src/emissions_allocation/config.py` and `src/emissions_allocation/selection.py` turn configured vessels into the fixed scope for every downstream checkpoint.


In [3]:
selection_rows = []
for vessel in cfg:
    selection_rows.append({
        "IMO": vessel.imo,
        "ship": vessel.shipnames[0],
        "label": vessel.label,
        "flag": vessel.require_spec("flag"),
        "type": vessel.require_spec("ship_type"),
        "DWT (t)": vessel.require_spec("dwt"),
        "names used by presence pull": ", ".join(vessel.shipnames),
    })
display(pd.DataFrame(selection_rows).style.format({"DWT (t)": "{:,.0f}"})
        .set_caption("Section 0: configured pilot vessels"))

for filename in ["vesselB_pool.json", "vesselB_shortlist.json"]:
    source = INTERIM / filename
    if source.exists():
        payload = json.loads(source.read_text(encoding="utf-8"))
        print(f"{filename}: retained discovery payload ({len(payload):,} top-level record(s))")


,IMO,ship,label,flag,type,DWT (t),names used by presence pull
0,9516454,COSCO ITALY,A,HKG,container,"156,610",COSCO ITALY
1,9277802,RCC AMERICA,B,BHS,vehicle,"21,182","RCC AMERICA, HOEGH AMERICA"


vesselB_pool.json: retained discovery payload (2,143 top-level record(s))
vesselB_shortlist.json: retained discovery payload (22 top-level record(s))


## 4. Stage 1: Acquire ship activity and identify unique ships

**Purpose:** Convert GFW presence reports into a single-vessel hourly track, then retain independently sourced port calls and derived voyage legs.

**Datasets and hand-off:** 
- GFW 4Wings `public-global-presence:latest` responses from `POST /v3/4wings/report` provide the hourly track. 
- GFW Events `public-global-port-visits-events:latest` responses from `GET /v3/events` provide port calls.
- GFW Vessels API provides the associated identity lookup.
- Raw JSON in `data/raw/` becomes `vessel_hour`, `port_call`, `voyage_leg`, and `coverage` checkpoints in `data/interim/`.

**Pipeline:** `src/emissions_allocation/gfw.py`, `src/emissions_allocation/activity.py`, `src/emissions_allocation/sql/12_voyage_leg.sql`, and `src/emissions_allocation/sql/13_coverage.sql`.

**Pipeline safeguards:** The processing asserts a non-empty pull, one distinct expected IMO, and observed-hour coverage before any modelling. This matters because an incorrect GFW ship name can otherwise return HTTP 200 with no rows.


In [4]:
activity_rows = []
for vessel in cfg:
    hours = read_checkpoint(f"vessel_hour_{vessel.imo}.parquet")
    calls = read_checkpoint(f"port_call_{vessel.imo}.parquet")
    legs = read_checkpoint(f"voyage_leg_{vessel.imo}.parquet")
    assert not hours.empty, f"{vessel.imo}: empty presence checkpoint"
    assert set(hours.imo.astype(str)) == {vessel.imo}, f"{vessel.imo}: identity integrity failure"
    activity_rows.append({
        "IMO": vessel.imo, "ship": vessel.shipnames[0], "vessel-hours": len(hours),
        "distinct IMO": hours.imo.astype(str).nunique(), "port calls": len(calls),
        "port countries": calls.port_iso3.nunique(), "voyage legs": len(legs),
        "international legs": int(legs.is_international.sum()),
    })
display(pd.DataFrame(activity_rows).style.format({"vessel-hours": "{:,}", "port calls": "{:,}",
                                                  "voyage legs": "{:,}", "international legs": "{:,}"})
        .set_caption("Section 1: identity and activity checkpoint assertions"))

preview_vessel = cfg.vessels[0]
hour_preview = read_checkpoint(f"vessel_hour_{preview_vessel.imo}.parquet")
display(hour_preview[["ts", "lat", "lon", "sog_raw", "sog_w3", "is_interpolated", "is_inactive"]]
        .head(6).style.format({"lat": "{:.3f}", "lon": "{:.3f}", "sog_raw": "{:.2f}", "sog_w3": "{:.2f}"})
        .set_caption("Section 1 input: hourly GFW presence transformed into speed and activity fields"))

call_preview = read_checkpoint(f"port_call_{preview_vessel.imo}.parquet")
display(call_preview[["start_ts", "end_ts", "duration_h", "port_id", "port_iso3", "confidence"]]
        .head(6).style.format({"duration_h": "{:.1f}"})
        .set_caption("Section 1 input: GFW port-event records"))

leg_preview = read_checkpoint(f"voyage_leg_{preview_vessel.imo}.parquet")
display(leg_preview[["depart_ts", "arrive_ts", "origin_iso3", "dest_iso3", "leg_hours", "is_eu_eu"]]
        .head(6).style.format({"leg_hours": "{:.1f}"})
        .set_caption("Section 1 output: voyage legs derived from consecutive port calls"))


,IMO,ship,vessel-hours,distinct IMO,port calls,port countries,voyage legs,international legs
0,9516454,COSCO ITALY,"70,128",1,389,17,388,225
1,9277802,RCC AMERICA,"70,128",1,588,64,587,395


,ts,lat,lon,sog_raw,sog_w3,is_interpolated,is_inactive
0,2017-01-01 00:00:00,9.800,66.310,16.96,16.96,False,False
1,2017-01-01 01:00:00,9.960,65.760,16.96,16.96,True,False
2,2017-01-01 02:00:00,9.960,65.760,16.96,14.43,False,False
3,2017-01-01 03:00:00,10.010,65.610,9.36,15.09,False,False
4,2017-01-01 04:00:00,10.090,65.300,18.95,14.75,False,False
5,2017-01-01 05:00:00,10.160,65.040,15.93,17.76,False,False


,start_ts,end_ts,duration_h,port_id,port_iso3,confidence
0,2017-01-07 19:30:04+00:00,2017-01-08 04:00:04+00:00,8.5,egy-suezsouthanchorage,EGY,4
1,2017-01-16 05:57:31+00:00,2017-01-18 14:41:16+00:00,56.7,nld-rotterdammaasvlakte,NLD,4
2,2017-01-19 11:37:51+00:00,2017-01-20 15:04:07+00:00,27.4,deu-hamburg,DEU,4
3,2017-01-20 19:18:17+00:00,2017-01-23 08:45:04+00:00,61.4,deu-deu-611,DEU,4
4,2017-01-24 03:56:19+00:00,2017-01-25 08:24:21+00:00,28.5,bel-antwerp,BEL,4
5,2017-01-25 14:39:20+00:00,2017-01-27 10:58:20+00:00,44.3,bel-antwerp,BEL,4


,depart_ts,arrive_ts,origin_iso3,dest_iso3,leg_hours,is_eu_eu
0,2017-01-07 23:00:04-05:00,2017-01-16 00:57:31-05:00,EGY,NLD,194.0,False
1,2017-01-18 09:41:16-05:00,2017-01-19 06:37:51-05:00,NLD,DEU,20.9,True
2,2017-01-20 10:04:07-05:00,2017-01-20 14:18:17-05:00,DEU,DEU,4.2,True
3,2017-01-23 03:45:04-05:00,2017-01-23 22:56:19-05:00,DEU,BEL,19.2,True
4,2017-01-25 03:24:21-05:00,2017-01-25 09:39:20-05:00,BEL,BEL,6.2,True
5,2017-01-27 05:58:20-05:00,2017-02-04 14:15:00-05:00,BEL,EGY,200.3,False


In [5]:
coverage = pd.concat([
    read_checkpoint(f"coverage_{vessel.imo}.parquet").assign(ship=vessel.shipnames[0])
    for vessel in cfg
], ignore_index=True)
assert coverage.coverage_active.between(0, 1).all()
display(coverage[["ship", "imo", "year", "elapsed_hours", "inactive_hours", "observed_hours",
                  "coverage_raw", "coverage_active"]]
        .style.format({"elapsed_hours": "{:,}", "inactive_hours": "{:,}", "observed_hours": "{:,}",
                       "coverage_raw": "{:.2%}", "coverage_active": "{:.2%}"})
        .set_caption("Section 1.7: observed coverage before emissions correction"))

bias_rows = []
for vessel in cfg:
    hours = read_checkpoint(f"vessel_hour_{vessel.imo}.parquet")
    active = hours.loc[~hours.is_inactive]
    for window in cfg.run["smoothing_windows"]:
        bias_rows.append({"IMO": vessel.imo, "ship": vessel.shipnames[0], "window (h)": window,
                          "mean(v³) / (mean(v))³": activity.cubic_bias(active[f"sog_w{window}"])})
display(pd.DataFrame(bias_rows).style.format({"mean(v³) / (mean(v))³": "{:.2f}x"})
        .set_caption("Section 1.6: cubic-speed bias by smoothing window"))


,ship,imo,year,elapsed_hours,inactive_hours,observed_hours,coverage_raw,coverage_active
0,COSCO ITALY,9516454,2017,"8,760",0,"7,185",82.02%,82.02%
1,COSCO ITALY,9516454,2018,"8,760","1,708","6,233",71.15%,88.39%
2,COSCO ITALY,9516454,2019,"8,760","4,906","3,159",36.06%,81.97%
3,COSCO ITALY,9516454,2020,"8,784","1,866","6,388",72.72%,92.34%
4,COSCO ITALY,9516454,2021,"8,760",0,"8,163",93.18%,93.18%
5,COSCO ITALY,9516454,2022,"8,760",0,"8,498",97.01%,97.01%
6,COSCO ITALY,9516454,2023,"8,760",0,"8,743",99.81%,99.81%
7,COSCO ITALY,9516454,2024,"8,784",0,"8,782",99.98%,99.98%
8,RCC AMERICA,9277802,2017,"8,760",0,"7,501",85.63%,85.63%
9,RCC AMERICA,9277802,2018,"8,760",0,"7,771",88.71%,88.71%


,IMO,ship,window (h),mean(v³) / (mean(v))³
0,9516454,COSCO ITALY,1,1.94x
1,9516454,COSCO ITALY,3,1.38x
2,9516454,COSCO ITALY,5,1.38x
3,9516454,COSCO ITALY,7,1.41x
4,9277802,RCC AMERICA,1,2.06x
5,9277802,RCC AMERICA,3,1.49x
6,9277802,RCC AMERICA,5,1.49x
7,9277802,RCC AMERICA,7,1.52x


## 5. Stage 2: Complete registry data and ship specifications

**Purpose:** Attach the vessel characteristics needed by the physical model, including power and design-speed estimates where observed values are unavailable.

**Dataset and hand-off:** Equasis supplies flag, gross tonnage, deadweight, ship type, build year, and company records. Public vessel registers supply dimensions where needed. The EEXI resolution, Charchalis calibration data, Cepowski and Chorab's beam relation, and IMO Fourth GHG Study tables support derived TEU, design-speed, installed-power, engine-type, and auxiliary-demand parameters. Configuration records each value with its unit, source, method, and estimated flag.

**Pipeline:** `src/emissions_allocation/specs.py`.


In [6]:
spec_rows = []
for vessel in cfg:
    for name, parameter in vessel.specs.items():
        spec_rows.append({
            "IMO": vessel.imo, "parameter": name, "value": parameter.value, "unit": parameter.unit,
            "estimated": parameter.estimated, "source": parameter.source, "method": parameter.method,
        })
spec_table = pd.DataFrame(spec_rows)
assert spec_table.loc[spec_table.estimated & spec_table.value.notna(), ["source", "method"]].notna().all().all()
display(spec_table.style.set_caption("Section 2: configured vessel specifications and provenance"))


,IMO,parameter,value,unit,estimated,source,method
0,9516454,mmsi,477845600,nan,False,GFW presence + Equasis,observed
1,9516454,callsign,VRNE4,nan,False,GFW registryInfo + Equasis,observed
2,9516454,flag,HKG,nan,False,Equasis; GFW registryInfo.flag,observed
3,9516454,ship_type,container,nan,False,Equasis,observed
4,9516454,year_built,2014,nan,False,Equasis,observed
5,9516454,dwt,156610,t,False,Equasis,observed
6,9516454,gt,154592,t,False,Equasis (since 2023),observed
7,9516454,gt_gfw,153666,t,False,GFW registryInfo.tonnageGt,observed
8,9516454,loa_m,365.900000,nan,False,public vessel registers,observed
9,9516454,beam_m,51.200000,nan,False,public vessel registers,observed


## 6. Stage 3: Assign fuel type and emission factors

**Purpose:** Establish each observed hour's fuel type and the corresponding IMO emission factor before calculating fuel consumption.

**Datasets and hand-off:** `vessel_hour` positions are joined to Marine Regions World EEZ v12, MARPOL Annex VI Regulation 14 ECA polygons, and Marine and Land Zones v4. GFW port events and derived `voyage_leg` records determine the EU-to-EU rule. IMO Fourth GHG Study Tables 19 and 21 supply fuel-specific SFC and CO₂ emission factors. `fuel.py` and SQL stages `20` through `30` write a one-row-per-vessel-hour `fuel_assignment` checkpoint.

**Pipeline:** `src/emissions_allocation/fuel.py` and the spatial and fuel SQL stages: `src/emissions_allocation/sql/20_eez_join.sql`, `21_eca_join.sql`, `22_distance_to_port.sql`, `23_distance_to_coast.sql`, `24_port_visit_hour.sql`, and `30_fuel_assignment.sql`.

The displayed check verifies that each activity hour receives one and only one fuel assignment.


In [7]:
fuel_rows = []
for vessel in cfg:
    fuel = read_checkpoint(f"fuel_assignment_{vessel.imo}.parquet")
    hours = read_checkpoint(f"vessel_hour_{vessel.imo}.parquet")
    assert len(fuel) == len(hours), f"{vessel.imo}: fuel assignment does not cover every vessel-hour"
    assert fuel.fuel_type.notna().all(), f"{vessel.imo}: unassigned fuel type"
    fuel_rows.append({
        "IMO": vessel.imo, "ship": vessel.shipnames[0], "hours": len(fuel),
        "ECA hours": int(fuel.in_eca.sum()), "EU-EU-leg hours": int(fuel.is_eu_eu_leg.sum()),
        "MDO/MGO hours": int((fuel.fuel_type == "mdo").sum()), "HFO hours": int((fuel.fuel_type == "hfo").sum()),
    })
display(pd.DataFrame(fuel_rows).style.format({c: "{:,}" for c in ["hours", "ECA hours", "EU?EU-leg hours", "MDO/MGO hours", "HFO hours"]})
        .set_caption("Section 3: fuel-assignment coverage and triggers"))

preview_vessel = cfg.vessels[0]
activity_for_fuel = read_checkpoint(f"vessel_hour_{preview_vessel.imo}.parquet")
fuel_for_preview = read_checkpoint(f"fuel_assignment_{preview_vessel.imo}.parquet")
fuel_preview = activity_for_fuel[["imo", "ts", "lat", "lon", "sog_w3"]].merge(
    fuel_for_preview[["imo", "ts", "in_eca", "eca_area", "is_eu_eu_leg", "fuel_type"]],
    on=["imo", "ts"],
    validate="one_to_one",
)
display(fuel_preview.head(8).style.format({"lat": "{:.3f}", "lon": "{:.3f}", "sog_w3": "{:.2f}"})
        .set_caption("Section 3 transformation: vessel-hour positions and voyage context to fuel assignment"))


,IMO,ship,hours,ECA hours,EU-EU-leg hours,MDO/MGO hours,HFO hours
0,9516454,COSCO ITALY,"70,128","4,310",1532,0,0
1,9277802,RCC AMERICA,"70,128","8,489",3084,0,0


,imo,ts,lat,lon,sog_w3,in_eca,eca_area,is_eu_eu_leg,fuel_type
0,9516454,2017-01-01 00:00:00,9.800,66.310,16.96,False,nan,False,HFO
1,9516454,2017-01-01 01:00:00,9.960,65.760,16.96,False,nan,False,HFO
2,9516454,2017-01-01 02:00:00,9.960,65.760,14.43,False,nan,False,HFO
3,9516454,2017-01-01 03:00:00,10.010,65.610,15.09,False,nan,False,HFO
4,9516454,2017-01-01 04:00:00,10.090,65.300,14.75,False,nan,False,HFO
5,9516454,2017-01-01 05:00:00,10.160,65.040,17.76,False,nan,False,HFO
6,9516454,2017-01-01 06:00:00,10.350,64.360,18.41,False,nan,False,HFO
7,9516454,2017-01-01 07:00:00,10.350,64.360,16.21,False,nan,False,HFO


## 7. Stage 4: Calculate CO2 emissions

**Purpose:** Use the Fourth IMO GHG Study operating-mode, load, SFC, and emission-factor rules to convert each fuel-assigned hour to CO2.

**Datasets and hand-off:** `emissions.py` combines `vessel_hour`, the Equasis and derived vessel specifications, `fuel_assignment`, and IMO Fourth GHG Study Tables 16, 17, 19, 20, and 21. It writes scenario-keyed `emissions_hour` and `emissions_year` checkpoints, preserving alternative installed-power estimates and speed-smoothing windows.

**Pipeline:** `src/emissions_allocation/emissions.py` and `src/emissions_allocation/sql/40_operating_mode.sql`, `42_emissions_hour.sql`, and `43_emissions_year.sql`.

The sample below follows this hand-off at hourly grain, then checks that the annual aggregation is present.


In [8]:
hourly_rows = []
for vessel in cfg:
    hourly = read_checkpoint(f"emissions_hour_{vessel.imo}.parquet")
    expected_scenarios = len(vessel.resolve_power_estimates(cfg.run["power_estimates"])) * len(cfg.run["smoothing_windows"])
    assert hourly.scenario_id.nunique() == expected_scenarios, f"{vessel.imo}: incomplete scenario expansion"
    assert (hourly.fc_total_g >= 0).all() and (hourly.co2_tonnes >= 0).all()
    hourly_rows.append({"IMO": vessel.imo, "ship": vessel.shipnames[0], "hour-scenario rows": len(hourly),
                        "scenarios": hourly.scenario_id.nunique(), "operating modes": hourly.operating_mode.nunique()})
display(pd.DataFrame(hourly_rows).style.format({"hour-scenario rows": "{:,}"})
        .set_caption("Section 4: scenario-keyed hourly emissions assertions"))

sample = read_checkpoint(f"emissions_hour_{cfg.vessels[0].imo}.parquet")
sample = sample[(sample.power_estimate == "A") & (sample.smoothing_window == 3)].head(8)
display(sample[["ts", "operating_mode", "sog", "me_load", "fuel_type", "w_me_kw", "w_ae_kw", "w_bo_kw",
                "fc_total_g", "co2_tonnes"]].style.format({"sog": "{:.2f}", "me_load": "{:.3f}",
                                                           "w_me_kw": "{:,.0f}", "w_ae_kw": "{:,.0f}",
                                                           "w_bo_kw": "{:,.0f}", "fc_total_g": "{:,.0f}",
                                                           "co2_tonnes": "{:.3f}"})
        .set_caption("Section 4: representative hourly emissions records (vessel A, estimate A, w=3)"))


,IMO,ship,hour-scenario rows,scenarios,operating modes
0,9516454,COSCO ITALY,"493,184",8,5
1,9277802,RCC AMERICA,"274,308",4,5


,ts,operating_mode,sog,me_load,fuel_type,w_me_kw,w_ae_kw,w_bo_kw,fc_total_g,co2_tonnes
61648,2017-01-01 00:00:00,slow_transit,16.96,0.292,HFO,"19,858","2,050",0,"4,261,585",13.271
61649,2017-01-01 01:00:00,slow_transit,16.96,0.292,HFO,"19,858","2,050",0,"4,261,585",13.271
61650,2017-01-01 02:00:00,slow_transit,14.43,0.180,HFO,"12,225","2,050",0,"2,896,288",9.019
61651,2017-01-01 03:00:00,slow_transit,15.09,0.206,HFO,"13,986","2,050",0,"3,221,901",10.033
61652,2017-01-01 04:00:00,slow_transit,14.75,0.192,HFO,"13,054","2,050",0,"3,050,408",9.499
61653,2017-01-01 05:00:00,slow_transit,17.76,0.336,HFO,"22,805","2,050",0,"4,761,410",14.827
61654,2017-01-01 06:00:00,slow_transit,18.41,0.374,HFO,"25,382","2,050",0,"5,189,010",16.159
61655,2017-01-01 07:00:00,slow_transit,16.21,0.255,HFO,"17,349","2,050",0,"3,825,413",11.912


In [9]:
annual = read_checkpoint("emissions_year.parquet")
assert annual.co2_tonnes.notna().all() and (annual.co2_tonnes >= 0).all()
annual_audit = annual[(annual.power_estimate == "A") & (annual.smoothing_window == 3)]
display(annual_audit[["imo", "year", "modelled_hours", "coverage_active", "co2_tonnes_observed",
                      "co2_tonnes_corrected", "co2_tonnes", "is_low_confidence"]]
        .style.format({"modelled_hours": "{:,}", "coverage_active": "{:.2%}",
                       "co2_tonnes_observed": "{:,.0f}", "co2_tonnes_corrected": "{:,.0f}",
                       "co2_tonnes": "{:,.0f}"})
        .set_caption("Section 4.5: annual CO2 aggregation (estimate A, w=3)"))


,imo,year,modelled_hours,coverage_active,co2_tonnes_observed,co2_tonnes_corrected,co2_tonnes,is_low_confidence
1,9516454,2017,"8,760",82.02%,"90,033","109,768","109,768",True
9,9516454,2018,"7,052",88.39%,"65,704","74,338","74,338",True
17,9516454,2019,"3,854",81.97%,"28,120","34,307","34,307",True
25,9516454,2020,"6,918",92.34%,"71,446","77,374","77,374",True
33,9516454,2021,"8,760",93.18%,"93,890","100,757","100,757",True
41,9516454,2022,"8,760",97.01%,"87,887","90,597","90,597",False
49,9516454,2023,"8,760",99.81%,"72,200","72,340","72,340",False
57,9516454,2024,"8,784",99.98%,"83,183","83,202","83,202",False
65,9277802,2017,"8,760",85.63%,"31,408","36,679","36,679",True
69,9277802,2018,"8,760",88.71%,"26,868","30,288","30,288",True


allocation.py writes allocation.parquet with every allocation option explicit.

In [10]:
domestic = pd.read_csv(OUT / "domestic_test.csv")
assert domestic.is_international.all(), "A configured pilot vessel is classified as domestic."
display(domestic.style.format({"dominant_eez_hours": "{:,}", "hours_in_any_eez": "{:,}", "hours_disputed": "{:,}", "dominant_eez_share": "{:.2%}"}))
keys = allocation.summarise_options(cfg)
display(keys.style.set_caption("Section 5.1: allocation keys under the paper-aligned country map"))
allocation_preview = read_checkpoint("allocation.parquet")
allocation_preview = allocation_preview[(allocation_preview.year == 2024) & (allocation_preview.power_estimate == "A") & (allocation_preview.smoothing_window == 3)]
display(allocation_preview[["option", "country", "gcb_name", "co2_tonnes", "n_vessels"]].sort_values(["option", "country"]).style.format({"co2_tonnes": "{:,.0f}"}))


,imo,dominant_eez_iso3,dominant_eez_hours,hours_in_any_eez,hours_disputed,dominant_eez_share,is_domestic,is_international
0,9277802,JPN,"3,609","50,619.0",111.0,7.13%,False,True
1,9516454,CHN,"11,825","39,481.0",533.0,29.95%,False,True


option,imo,flag,manager,operator,owner,n_distinct_countries,is_degenerate
0,9277802,BHS,GRC,IMN,IMN,3,False
1,9516454,HKG,CHN,CHN,CHN,1,True


,option,country,gcb_name,co2_tonnes,n_vessels
93,flag,BHS,Bahamas,"20,322",1
89,flag,HKG,China,"83,202",1
185,manager,CHN,China,"83,202",1
189,manager,GRC,Greece,"20,322",1
281,operator,CHN,China,"83,202",1
285,operator,IMN,United Kingdom,"20,322",1
377,owner,CHN,China,"83,202",1
381,owner,IMN,United Kingdom,"20,322",1


It converts the reported MtC values to MtCO? and uses the fixed country map derived from Selin et al. supplementary Table 1.

In [11]:
baseline = read_checkpoint("baseline.parquet")
allocation_result = read_checkpoint("allocation.parquet")
impacts = read_checkpoint("impacts.parquet")
assert baseline.mtco2.notna().all()
assert impacts.baseline_mt.notna().all()
assert impacts.delta_e_pct.notna().all()
base_2024 = baseline[(baseline.year == 2024) & (baseline.country == "China")]
display(base_2024.style.format({"mtc": "{:,.2f}", "mtco2": "{:,.2f}"}).set_caption("Section 6: 2024 China GCB baseline under the paper-aligned map"))
impact_2024 = impacts[(impacts.year == 2024) & (impacts.power_estimate == "A") & (impacts.smoothing_window == 3)]
display(impact_2024[["option", "country", "gcb_name", "delta_e_mt", "baseline_mt", "delta_e_pct"]].sort_values(["option", "country"]).style.format({"delta_e_mt": "{:.5f}", "baseline_mt": "{:,.1f}", "delta_e_pct": "{:.6f}"}))


,year,country,mtc,mtco2
311,2024,China,"3,353.99","12,289.04"


,option,country,gcb_name,delta_e_mt,baseline_mt,delta_e_pct
87,flag,BHS,Bahamas,0.02032,3.1,0.662009
86,flag,HKG,China,0.08320,"12,289.0",0.000677
182,manager,CHN,China,0.08320,"12,289.0",0.000677
183,manager,GRC,Greece,0.02032,53.4,0.038083
278,operator,CHN,China,0.08320,"12,289.0",0.000677
279,operator,IMN,United Kingdom,0.02032,312.9,0.006494
374,owner,CHN,China,0.08320,"12,289.0",0.000677
375,owner,IMN,United Kingdom,0.02032,312.9,0.006494


## 10. Stage 8: Sensitivity and validation

**Purpose:** Show how the uncertainty from power and speed smoothing propagates to final impacts, then retain evidence from structural and external checks.

**Datasets and hand-off:** `scenario_spread.parquet` summarizes the power-estimate and smoothing-window scenarios after they reach the impact calculation. Per-vessel validation CSVs use activity, voyage, fuel, and emissions checkpoints for internal checks. EU THETIS-MRV verified annual CO2 is used only as external validation, never as a model input. A warning or failure remains evidence for follow-up.

**Pipeline:** `src/emissions_allocation/validate.py`.


In [12]:
spread = read_checkpoint("scenario_spread.parquet")
valid_scenario_counts = {len(vessel.resolve_power_estimates(cfg.run["power_estimates"])) * len(cfg.run["smoothing_windows"]) for vessel in cfg}
assert spread.n_scenarios.isin(valid_scenario_counts).all()
display(spread[spread.year == 2024][["option", "country", "delta_e_mt_min", "delta_e_mt_max", "spread_ratio", "n_scenarios"]].style.format({"delta_e_mt_min": "{:.5f}", "delta_e_mt_max": "{:.5f}", "spread_ratio": "{:.2f}x"}))


,option,country,delta_e_mt_min,delta_e_mt_max,spread_ratio,n_scenarios
7,flag,BHS,0.01947,0.02177,1.12x,4
15,flag,HKG,0.07964,0.13301,1.67x,8
23,manager,CHN,0.07964,0.13301,1.67x,8
31,manager,GRC,0.01947,0.02177,1.12x,4
39,operator,CHN,0.07964,0.13301,1.67x,8
47,operator,IMN,0.01947,0.02177,1.12x,4
55,owner,CHN,0.07964,0.13301,1.67x,8
63,owner,IMN,0.01947,0.02177,1.12x,4
